# Imports

In [1]:
import numpy as np
import itertools
import functools
from numpy import random as random
import jax
import jax.numpy as jnp

from functools import partial
from scipy.io import loadmat

import matplotlib as mpl
from matplotlib import pyplot as plt

In [2]:
rng = random.default_rng()
key = jax.random.key(42)

In [3]:
movieBinnedSpiking = loadmat('../Data/movieBinnedSpiking.mat')

### Let's start with constant h and see what happens

In [30]:
def corrs(sigma):
    ''' 
    Input:
        - sigma: a (B,N) matrix of the response of N neurons at B time indices, where sigmat[b,n] corresponds to the response of the nth neuron at time b

    Output:
        - corrs: a (B,N*(N-1)/2) matrix of the correlation of the N neurons at B indices, for all corrs[b,i*j/2] where i < j
    '''

    return jnp.apply_along_axis(func1d=lambda x : jnp.outer(x,x)[jnp.tril_indices(x.shape[0]-1)],axis=1,arr=sigma)

def pt(sigma,**X):
    '''
    Input:
        - sigma: a (N+N*(N-1)/2,) vector of the response of N neurons at a particular time t, where sigma[i] corresponds to the response of the ith neuron, concatenated with the cross-correlations of each neuron at that time
            - sigma can also be (B,N+N*(N-1)/2)
        - X: a (N + N*(N-1)/2 , ) vector containing:
            - h: (B*N,) vector of the time-dependent field, where h[N*t + n] corresponds to the time dependent field of the nth neuron at time t
            - J: (N(N-1)/2,) vector of the fixed couplings between two neurons

    Output:
        pt: the probability of that state at that time, given the parameters
    '''
    
    if 'X' in X.keys():
        X_ = X['X']
    else:
        X_ = jnp.concatenate((X['h'],X['J']))
    
    if 'Z' in X.keys():
        Z_ = X['Z']
    else:
        Z_ = 1

    pt = jnp.exp(-1*jnp.matmul(sigma,X_)) / Z_
    
    return pt

def observables(sigma):
    '''
    Input:
        - sigma: an (B,N) vector of binarized neural responses at time b; sigmab[n] is the response of neuron n
    
    Output:
        - observables: (B, N + N*(N-1)/2) vector of the observables at that point in time
    '''
    corr = corrs(sigma)
    return jnp.concatenate((sigma,corr),axis=1)

def P_bar(sigma):
    ''' 
    Input:
        sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b

    Output:
        P_b: a (N + N*(N-1)/2 , ) vector of the average of the observables given the neural data
    '''
    P_b = observables(sigma).mean(axis=0)

    return P_b

def Q(X,combs_obs,Z):
    '''
    Input:
        - X: a (N + N*(N-1)/2 , ) vector of the parameters of the model
    
    Output:
        Q: a (N + N*(N-1)/2 , ) vector of the model averages of the observables.
    '''

    weighted_observables = combs_obs*pt(combs_obs,X=X).reshape(-1,1)

    Q = weighted_observables.sum(axis=0)

    return Q

def QMC(sigma,M):
    '''
    Input:
        - sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b
        - M: the number of times to sample from sigma
    
    Output:
        QMC: a montecarlo approximation of Q
    '''
    B,N = sigma.shape
    MC = rng.choice(sigma,M,replace=True)
    QMC = observables(MC)
    QMC = QMC.mean(axis=0)

    return QMC

def susc_bar(sigma):
    '''
    Input:
        - sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b
    
    Output:
        - X_bar: a (D,D) matrix consisting of the mean of the products of observables, subtracted from the product of the mean of each corresponding observable
            where D = N + N*(N-1)/2
    '''
    B,N = sigma.shape
    all_obs = observables(sigma)
    obs_prod_bar = jnp.apply_along_axis(lambda x: jnp.outer(x,x),axis=1,arr=all_obs).mean(axis=0)
    P_ = all_obs.mean(axis=0)
    prod_obs_bar = jnp.outer(P_,P_)
    return obs_prod_bar - prod_obs_bar

def epsilon(P_bar,inv_sus,Q,B):
    D = P_bar.shape[0]
    diff = P_bar - Q
    return jnp.sqrt(jnp.abs((2*B / D)*(diff@inv_sus@diff)))

def all_combs_obs(N):
    combs = jnp.array(np.fromiter(itertools.product(range(2),repeat = N),dtype=np.dtype((jnp.float32,N)),count=2**N))

    combs_obs = observables(combs)

    return combs_obs

def Z(X,combs_obs):
    p_ = pt(combs_obs,X=X)
    return p_.sum()

In [47]:
class fixed_h_MaxEnt:
    def __init__(
        self,
        X0_init = 'random',
        a0 = 1,
        del_p = np.float32(1.05),
        del_n = np.sqrt(2,dtype=np.float32)
    ):
        self.a0 = a0
        self.X0_init = 'random'
        self.del_p = del_p
        self.del_n = del_n
        self.X = None

    def fit(self,sigma):
        B,N = sigma.shape
        D = int(N+N*(N-1)/2)
        if self.X0_init == 'random':
            X0 = random.rand(D)
        else:
            print('not implemented yet!')
            return None


        combs_obs = all_combs_obs(N)
        Z0 = Z(X0,combs_obs)
        Q0 = Q(X0,combs_obs,Z0)
        P_ = P_bar(sigma)
        X_ = susc_bar(sigma)
        inv_X_ = jnp.linalg.pinv(X_,hermitian=True)
        e0 = epsilon(P_,inv_X_,Q0,B)

        et = e0
        Qt = Q0
        Xt = X0
        at = self.a0
        Zt = Z0

        while et >= 1:
            Mt = jnp.ceil(jnp.minimum(B/et**2,B)).astype(jnp.int32)
            del_Xt = at*inv_X_@(P_ - Qt)
            Xt_1 = Xt + del_Xt
            Qt_1 = QMC(sigma,Mt)
            et_1 = epsilon(P_,inv_X_,Qt_1,B)
            if et_1 < et:
                print(et_1)
                Qt = Qt_1
                Xt = Xt_1
                et = et_1
                at_1 = at*self.del_p
                at = at_1
            else:
                at_1 = at/self.del_n
                at = at_1
        
        Xf = Xt
        Zf = Zt
        ef = et
        self.X = Xf
        self.Z = Zf

        return Xf,ef
    
    def predict(self,sigmat):
        if self.X == None:
            print('Hasn\'t been fit yet!')
            return None
        return pt(sigmat,X=self.X,Z = self.Z)
            

In [48]:
B = 5000
N = 20
D = int(N + N*(N-1)/2)
t = 2
sigma = jax.random.choice(key,jnp.array([0,1]),(B,N),replace=True)
X = jax.random.uniform(key,D)
P_ = P_bar(sigma)
X_ = susc_bar(sigma)
obs = observables(sigma)

In [49]:
A = QMC(sigma,50)

In [50]:
Maxent = fixed_h_MaxEnt()

In [51]:
movnames = movieBinnedSpiking['movnames']
ncell = movieBinnedSpiking['ncell']
nmov = movieBinnedSpiking['nmov']
nreps = movieBinnedSpiking['nreps'].flatten()
samplingFreq = movieBinnedSpiking['samplingFreq']
binned = movieBinnedSpiking['binned']

In [52]:
mov1 = binned[1:nreps[0],:,:,0]

In [53]:
mov1_2d = mov1.reshape((-1,93))
idx = rng.choice(93,20,replace=False)
mov1_sigma = mov1_2d[30:,idx]
mov1_sigma = jnp.array(mov1_sigma)

In [ ]:
Maxent.fit(mov1_sigma)

10.916148
9.468442
8.642272
8.452108
7.563279
7.5354795
6.8994985
6.7109838
5.941772
5.903641
5.896221
5.8804803
5.6889405
5.4466524
5.2451773
5.231679
